# 🎯 [DAT06_M03] 표준 정산 규칙 기반 1차 월간 결산 집계 및 통계 산출 Baseline 엔진 구현
### Google Colab 실습 환경 (데이터·자동화·업무플랫폼 / Level 3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/DAT06/dat06_m03_student.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 데이터·자동화·업무플랫폼 (Level 3) |
| **미션 ID** | DAT06_M03 |
| **미션 주제** | 표준 정산 규칙 기반 1차 월간 결산 집계 및 통계 산출 Baseline 엔진 구현 |
| **소요 시간** | 약 2.5시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 배경 및 당위성
미국 국방재무회계국(DFAS)은 반복적인 재무·정산 업무에 업무 자동화(RPA)를 도입해 사람이 하던 취합과 점검을 줄이는 사업을 추진해 왔다. 이 공개 사례의 학습 초점은 반복 업무 분해, 입력과 트리거, 자동 처리 순서, 오류 복구, 실행 로그, 시간 절감이다. 본 프로젝트는 이를 바탕으로 교육용 가상 부대 24개의 월간 결산 자료(물자 불출, 수리부속 소모, 탄약 정산, 장비 가동률)를 코드로 만들고, 파일 대신 메모리 안의 가상 문서와 표로 모의한다. 학습자는 업무 분해, 다형식 파싱, 검증 룰 엔진, 이상 원인 추천, 일관성 검증, 승인·반려 대시보드, 3단계 사람 결재선, 대량 배치 시험, AAR 보정, 최종 통합을 차례로 수행한다. 최종 목표는 월간 정산·보고·취합을 한 번에 실행하되 발간은 반드시 사람이 결재하도록 하는 원클릭 자동화 도구를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

> 💡 **[산출물(Deliverables) 디렉토리 경로 및 실습 안내]**
> - **아키텍처 관점의 표준 산출물:** 과제 명세에 표기된 src/connectors/..., docs/..., dist/... 등의 경로는 본 솔루션이 실무 배포 환경에서 갖추게 되는 **소프트웨어 표준 아키텍처 구조**를 의미합니다.
> - **노트북 중심 실습:** 별도의 폴더나 외부 파이썬 파일(.py)을 수작업으로 생성하실 필요가 없습니다. 본 노트북의 각 미션별 실습 셀을 AI 코파일럿과 함께 완성하고 검증하면 해당 산출물 요건이 자동으로 충족됩니다.
> - **미션 10 최종 통합 패키징:** 각 미션 셀에서 구현된 단위 모듈과 명세는 **[미션 10]** 실행 시 단일 소프트웨어 패키지(dist/dat06_package/)와 운영 매뉴얼(docs/dat06_final_manual.md)로 일괄 결합되어 자동 추출됩니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 03. 표준 정산 규칙 기반 1차 월간 결산 집계 및 통계 산출 Baseline 엔진 구현
**소요시간**: 약 2.5시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

표준 정산 규칙 기반 1차 월간 결산 집계 Baseline 엔진을 구현한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **표준 정산 규칙을 설정 문자열 룰 15종으로 분리하고, 이를 적용한 1차 월간 결산 집계 Baseline 엔진을 구현한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **룰마다 가상 조항 번호, 조건식, 오류 메시지를 설정으로 두어 코드 수정 없이 바꿀 수 있게 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/engine/dat06_rule_engine.py` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 군 규정 및 지침 조항과의 정확한 일치성

#### 미션 PASS 조건

- 정량 기준: 업무 검증 룰 15종 이상 코딩, 룰 처리 지연시간 50ms 이하


- 진행 기준: 룰 엔진 단위 테스트 100% 통과


- 검증 방법: 경계 조건(Boundary condition)에 대한 룰 판정 검증


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""설정 문자열로 분리한 검증 룰 15종(가상 정산 규정 조항 대응)으로 월간 원장을 검사하고, 표준 정산 규칙에 따른 1차 결산 집계 Baseline을 계산하며 처리 지연시간과 경계 조건 단위 테스트를 확인한다."""
import time
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 월간 결산 원장 생성기 (실제 부대명·품목·수량·재무 정보 없음)
# 가상 대대 24개 x (물자 불출 / 수리부속 소모 / 탄약 정산) 품목 원장과 장비 가동 현황을 만든다.
# 재고 산식: 기말 = 기초 + 수령 + 반납 - 불출,  다음 달 기초 = 이번 달 기말.
# ---------------------------------------------------------------------------
UNITS = [f"가상{i:02d}대대" for i in range(1, 25)]
ITEMS = {"물자불출": ["급식류", "피복류", "유류", "정비소모품"],
         "수리부속": ["필터류", "베어링류", "전장품", "타이어류"],
         "탄약정산": ["훈련탄A", "훈련탄B", "모의탄"]}
UNIT_CODE = {"급식류": "BX", "피복류": "EA", "유류": "L", "정비소모품": "EA", "필터류": "EA", "베어링류": "EA",
             "전장품": "EA", "타이어류": "EA", "훈련탄A": "RD", "훈련탄B": "RD", "모의탄": "RD"}
LEDGER_COLS = ["month", "unit", "cat", "item", "open", "recv", "issue", "ret", "close"]


def make_ledger(n_months=2, seed=60):
    """n_months개월 원장(list of dict). 둘째 달 기초는 첫째 달 기말과 같다."""
    rng = np.random.default_rng(seed)
    rows, prev = [], {}
    for m in range(n_months):
        for u in UNITS:
            for cat, items in ITEMS.items():
                for it in items:
                    base = int(rng.integers(80, 400))
                    op = prev.get((u, it), base * 3)
                    recv = int(rng.integers(0, base))
                    issue = int(rng.integers(base // 4, base // 2 + 1))
                    ret = int(issue * rng.uniform(0, 0.15))
                    cl = op + recv + ret - issue
                    rows.append({"month": f"2026-{m + 7:02d}", "unit": u, "cat": cat, "item": it,
                                 "open": op, "recv": recv, "issue": issue, "ret": ret, "close": cl})
                    prev[(u, it)] = cl
    return rows


def make_equip(n_months=2, seed=61):
    rng = np.random.default_rng(seed)
    out = []
    for m in range(n_months):
        for u in UNITS:
            hold = int(rng.integers(20, 60))
            out.append({"month": f"2026-{m + 7:02d}", "unit": u, "hold": hold, "operable": int(hold * rng.uniform(0.78, 0.97))})
    return out

# 1) 검증 룰 설정: (룰ID, 가상 조항, 조건식(참이어야 정상), 오류 메시지) - 코드 수정 없이 문자열만 바꿔 룰 변경
RULES = [
    ("V01", "가상규정 제1조", "unit in UNITS", "존재하지 않는 부대"),
    ("V02", "가상규정 제2조", "item in UNIT_CODE", "알 수 없는 품목"),
    ("V03", "가상규정 제3조", "cat in ITEMS and item in ITEMS[cat]", "구분과 품목 불일치"),
    ("V04", "가상규정 제4조", "open >= 0", "기초 재고 음수"),
    ("V05", "가상규정 제4조", "recv >= 0", "수령량 음수"),
    ("V06", "가상규정 제4조", "issue >= 0", "불출량 음수"),
    ("V07", "가상규정 제4조", "ret >= 0", "반납량 음수"),
    ("V08", "가상규정 제5조", "close == open + recv + ret - issue", "기말 산식 불일치"),
    ("V09", "가상규정 제6조", "issue <= open + recv", "가용량 초과 불출"),
    ("V10", "가상규정 제6조", "ret <= issue", "불출보다 큰 반납"),
    ("V11", "가상규정 제7조", "close >= 0", "기말 재고 음수"),
    ("V12", "가상규정 제8조", "len(month) == 7 and month[4] == '-'", "결산월 형식 오류"),
    ("V13", "가상규정 제9조", "cat != '탄약정산' or issue <= 5000", "탄약 월 사용량 한도 초과"),
    ("V14", "가상규정 제10조", "isinstance(open, int) and isinstance(close, int)", "수량 자료형 오류"),
    ("V15", "가상규정 제11조", "issue <= max(open + recv, 1) * 0.9 or issue == 0", "불출 비율 과다(90% 초과)"),
]
COMPILED = [(rid, art, compile(expr, rid, "eval"), msg) for rid, art, expr, msg in RULES]
CTX = {"UNITS": UNITS, "UNIT_CODE": UNIT_CODE, "ITEMS": ITEMS}


def check_row(row):
    # [TODO : 학생 실습 3] 표준 정산 규칙 기반 1차 월간 결산 집계 및 통계 산출 Baseline 엔진 구현 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
rows = make_ledger(2)
rng = np.random.default_rng(3)
for i in rng.choice(len(rows), 6, replace=False):          # 오류 6건 주입(룰이 잡는지 확인)
    k = int(rng.integers(0, 3))
    rows[i] = {**rows[i], **[{"close": rows[i]["close"] + 7}, {"issue": rows[i]["open"] + rows[i]["recv"] + 50}, {"ret": -3}][k]}
lat, found = [], []
for r in rows:
    t0 = time.perf_counter()
    res = check_row(r)
    lat.append((time.perf_counter() - t0) * 1000)
    found += [(r["month"], r["unit"], r["item"], *x) for x in res]
lat = np.array(lat)
print(f"[검증 룰 {len(RULES)}종 적용] 대상 {len(rows)}행, 위반 {len(found)}건 [교육용가상자료]")
cnt = pd.Series([f[3] for f in found]).value_counts()
print("룰별 위반 건수:", {k: int(v) for k, v in cnt.items()})
print("위반 예시(3건):")
for f in found[:3]:
    print("  ", f)
print(f"룰 처리 지연: 평균 {lat.mean():.3f}ms, 최대 {lat.max():.3f}ms (목표 50ms 이하: {'달성' if lat.max() <= 50 else '미달'}, 환경에 따라 다름)")

# 3) 1차 결산 집계 Baseline: 부대 x 구분 합계, 구분별 통계
clean = pd.DataFrame([r for r in make_ledger(2) if r["month"] == "2026-08"])
pv = clean.pivot_table(index="unit", columns="cat", values="issue", aggfunc="sum")
print("\n[1차 결산 집계 - 8월 불출·소모량 부대별(상위 4개)]")
print(pv.head(4).to_string())
st = pv.agg(["sum", "mean", "std"]).round(1)
print("[구분별 합계·평균·표준편차]")
print(st.to_string())
print(f"전체 기말 재고 합계 {int(clean['close'].sum())}, 보고 대상 부대 {clean['unit'].nunique()}개")

# 4) 경계 조건 단위 테스트
base = {"month": "2026-08", "unit": UNITS[0], "cat": "물자불출", "item": "유류", "open": 100, "recv": 0, "issue": 0, "ret": 0, "close": 100}
tests = [("정상 행", base, []),
         ("불출 = 가용량 정확히 일치(V15만 위반)", {**base, "issue": 100, "close": 0}, ["V15"]),
         ("불출 = 가용량+1 (V09, V11, V15)", {**base, "issue": 101, "close": -1}, ["V09", "V11", "V15"]),
         ("기말 0 허용", {**base, "issue": 90, "close": 10, "ret": 0}, []),
         ("반납 = 불출 경계 허용", {**base, "issue": 50, "ret": 50, "close": 100}, []),
         ("탄약 한도 5000 경계", {**base, "cat": "탄약정산", "item": "훈련탄A", "open": 9000, "issue": 5000, "close": 4000}, []),
         ("탄약 한도 5001 초과", {**base, "cat": "탄약정산", "item": "훈련탄A", "open": 9000, "issue": 5001, "close": 3999}, ["V13"]),
         ("결산월 형식 오류", {**base, "month": "202608"}, ["V12"]),
         ("자료형 오류(문자열 수량)", {**base, "open": "100"}, ["V14"])]   # 이 경우 V14 포함 여부만 확인
ok = 0
print("\n[경계 조건 단위 테스트]")
for name, r, exp in tests:
    got = sorted(x[0] for x in check_row(r))
    p = set(exp) <= set(got) if name.startswith("자료형") else got == exp
    ok += p
    print(f" - {name}: 기대 {exp} / 결과 {got} -> {'PASS' if p else 'FAIL'}")
print(f"단위 테스트 {ok}/{len(tests)} 통과({ok / len(tests):.0%})")
print(f"목표(룰 15종 이상, 지연 50ms 이하, 단위 테스트 100%): {'달성' if len(RULES) >= 15 and lat.max() <= 50 and ok == len(tests) else '미달'}")


<details>
<summary><b>💡 [참고] 미션 03 예상 실행 결과 (Output)</b></summary>

```text
[검증 룰 15종 적용] 대상 528행, 위반 13건 [교육용가상자료]
룰별 위반 건수: {'V08': 6, 'V09': 3, 'V15': 3, 'V07': 1}
위반 예시(3건):
   ('2026-07', '가상05대대', '급식류', 'V08', '가상규정 제5조', '기말 산식 불일치')
   ('2026-07', '가상05대대', '급식류', 'V09', '가상규정 제6조', '가용량 초과 불출')
   ('2026-07', '가상05대대', '급식류', 'V15', '가상규정 제11조', '불출 비율 과다(90% 초과)')
룰 처리 지연: 평균 0.008ms, 최대 0.072ms (목표 50ms 이하: 달성, 환경에 따라 다름)

[1차 결산 집계 - 8월 불출·소모량 부대별(상위 4개)]
cat     물자불출  수리부속  탄약정산
unit                    
가상01대대   378   385   384
가상02대대   255   427   234
가상03대대   239   371   168
가상04대대   305   258   319
[구분별 합계·평균·표준편차]
cat     물자불출    수리부속    탄약정산
sum   8270.0  8184.0  6743.0
mean   344.6   341.0   281.0
std     79.3    72.2    75.9
전체 기말 재고 합계 199291, 보고 대상 부대 24개

[경계 조건 단위 테스트]
 - 정상 행: 기대 [] / 결과 [] -> PASS
 - 불출 = 가용량 정확히 일치(V15만 위반): 기대 ['V15'] / 결과 ['V15'] -> PASS
 - 불출 = 가용량+1 (V09, V11, V15): 기대 ['V09', 'V11', 'V15'] / 결과 ['V09', 'V11', 'V15'] -> PASS
 - 기말 0 허용: 기대 [] / 결과 [] -> PASS
 - 반납 = 불출 경계 허용: 기대 [] / 결과 [] -> PASS
 - 탄약 한도 5000 경계: 기대 [] / 결과 [] -> PASS
 - 탄약 한도 5001 초과: 기대 ['V13'] / 결과 ['V13'] -> PASS
 - 결산월 형식 오류: 기대 ['V12'] / 결과 ['V12'] -> PASS
 - 자료형 오류(문자열 수량): 기대 ['V14'] / 결과 ['V04', 'V08', 'V09', 'V14', 'V15'] -> PASS
단위 테스트 9/9 통과(100%)
목표(룰 15종 이상, 지연 50ms 이하, 단위 테스트 100%): 달성
```
</details>
